# Step by Step Guide to use this GraphSage Implementation

This notebook provides a step-by-step guide to using the GraphSage implementation provided in this repository. It covers the necessary imports, model setup, and training process. One can follow along to understand how to utilize the implemented GraphSage algorithm for node classification tasks. If one just wants to run the code, they can run python main.py with the appropriate arguments, but this notebook is meant to provide a deeper understanding of the code and its components.

Before starting, make sure to have the necessary libraries installed and the dataset ready. The code is structured to be modular, allowing for easy modifications and experimentation with different configurations. One can install the required libraries using pip install -r requirements.txt and ensure that the dataset is in the correct format and location as expected by the code.

## Setup 

In [17]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import torch.nn.functional as F
from src.Algo_Mini_Batch import AlgoMiniBatch
from src.train import train
from src.layers import MeanAggregator, MaxPoolingAggregator, LSTMAggregator

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


### Choice and loading of the graph dataset

Three graphs are avaible by default: ppi, reddit and cora. You can also generate your own graph dataset using the script in src/generation_dataset/generate_graph_dataset.py and following the instructions in the README.md file.

In [19]:
GRAPH_PATH = "src/generation_dataset/2026-02-10_14-21-11/graphs/ppi.pt"

G = torch.load(GRAPH_PATH, weights_only=False)

print("\nGraph loaded")
print("Nb nodes :", G.number_of_nodes())
print("Nb edges :", G.number_of_edges())

node0 = list(G.nodes())[0]
assert "x" in G.nodes[node0], "ERREUR : pas de features 'x'"
print(f"Exemple de feature : {G.nodes[node0]['x']}")

feat_dim = G.nodes[node0]["x"].shape[0]

for node in G.nodes():
    G.nodes[node]["features"] = G.nodes[node]["x"]


Graph loaded
Nb nodes : 1767
Nb edges : 32318
Exemple de feature : tensor([-0.0855, -0.0884, -0.1128, -0.1719, -0.0766, -0.1003, -0.0751, -0.1149,
        -0.1212, -0.0994,  0.0000, -0.1699, -0.0428, -0.1123, -0.0760, -0.1152,
        -0.1031, -0.1120, -0.1435, -0.0975, -0.0875, -0.1457, -0.1234, -0.1242,
        -0.0976, -0.1197, -0.1161, -0.0735, -0.0667, -0.0873, -0.1797, -0.1447,
        -0.1606, -0.1582, -0.1477, -0.4350, -0.1617, -0.1556, -0.1526, -0.1396,
        -0.1281, -0.1539, -0.1593, -0.1546, -0.1466, -0.1449, -0.1568, -0.1399,
        -0.1494, -0.1481])


### Aggregator choice

Three aggregators are implemented and can be used: MeanAggregator, MaxPoolingAggregator and LSTMAggregator. 

In [20]:
aggregator = str(input("Choose an aggregator (MeanAggregator, MaxPoolingAggregator, LSTMAggregator) : "))

depth = 2
hidden_dim = feat_dim

if aggregator == "mean":
    agg = [MeanAggregator() for _ in range(depth)]
elif aggregator == "max":
    agg = [MaxPoolingAggregator() for _ in range(depth)]
elif aggregator == "lstm":
    agg = [LSTMAggregator(hidden_dim) for _ in range(depth)]

## Model and training

In [21]:
sample_size = 5

W = nn.ModuleList([
    nn.Linear(2 * hidden_dim, hidden_dim)
    for _ in range(depth)
])

sample_size = [sample_size for _ in range(depth)]

model = AlgoMiniBatch(depth, W, F.relu, agg).to(device)

print("\nNb paramètres :", sum(p.numel() for p in model.parameters()))


Nb paramètres : 10100


In [22]:
print("\n--------------------------------")
print("La phase d'entrainement débute..")
EPOCHS = 10
BATCH_SIZE = 128
train(
    model,
    G,
    device,
    sampling_size=sample_size,
    epochs=EPOCHS,
    learning_rate=3e-4,
    batch_size=BATCH_SIZE
)

print("\nL'entrainement est terminé.")


--------------------------------
La phase d'entrainement débute..
--------------------------------

Epoque 1 / 10


Average loss: 17.7847

Epoque 2 / 10


Average loss: 14.7458

Epoque 3 / 10


Average loss: 14.6307

Epoque 4 / 10


Average loss: 14.6058

Epoque 5 / 10


...

Average loss: 14.5810

Epoque 7 / 10


Average loss: 14.5769

Epoque 8 / 10


Average loss: 14.5757

Epoque 9 / 10


Average loss: 14.5764

Epoque 10 / 10


Average loss: 14.5744
Entrainement terminé.

L'entrainement est terminé.


In [23]:
model.eval()

all_nodes = list(G.nodes())

with torch.no_grad():
    embeddings = model.forward_propagation(G, all_nodes, sample_size)

print("Shape embeddings :", embeddings.shape)
for i in range(50):
    print(embeddings[i])

Shape embeddings : torch.Size([1767, 50])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.